# PRA Cache Services Experiment

Interactive mirror of `tests/test_cache_services.py`: inspect metadata extraction, configurable resolver/cache services, and single-sample cache construction.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
if repo.name == "nb":
    repo = repo.parent
sys.path.insert(0, str(repo / "src"))
sys.path.insert(0, str(repo))

from data.datamodules import PRADataModule
from pra_torch.cache_services import build_cache_from_metadata, collect_reference_metadata, create_cache, create_resolver
from pra_torch.config import CacheServiceConfig, PRAConfig, ResolverServiceConfig
from pra_torch.model import TinyPRAModel

## Build A Batch

In [ ]:
dm = PRADataModule("stage0_synthetic_memory", repo / "data", max_examples=1, batch_size=1, max_seq_len=64).load()
batch = next(iter(dm.train_loader()))
model = TinyPRAModel(PRAConfig(vocab_size=dm.tokenizer.vocab_size, max_seq_len=64, d_model=32, n_heads=4, n_layers=2))

{
    "input_shape": tuple(batch["input_ids"].shape),
    "references": [ref.uri for ref in batch["metadata"][0]["references"]],
}

## Inspect Resolver Inputs

In [ ]:
documents, summaries, handles = collect_reference_metadata(batch["metadata"])
{
    "document_uris": sorted(documents),
    "summary_uris": sorted(summaries),
    "handle_tokens": [handle.token for handle in handles],
}

## Build Cache With Service Configs

In [ ]:
resolver_config = ResolverServiceConfig(type="in_memory")
cache_config = CacheServiceConfig(type="simple")
cache = build_cache_from_metadata(
    model,
    dm.tokenizer,
    batch["metadata"],
    "cpu",
    resolver_config=resolver_config,
    cache_config=cache_config,
)

{
    "model_owns_cache": model.pra_cache is cache,
    "cache_entries": len(cache.entries),
    "layer_kv": {uri: sorted(entry.layer_kv) for uri, entry in cache.entries.items()},
}

## Single-Sample Helper For Experiments

In [ ]:
def build_cache_for_example(model, tokenizer, sample, device, *, resolver_config=None, cache_config=None):
    return build_cache_from_metadata(
        model,
        tokenizer,
        [{"references": sample.references}],
        device,
        resolver_config=resolver_config,
        cache_config=cache_config,
    )

single_cache = build_cache_for_example(model, dm.tokenizer, dm.dataset[0], "cpu")
{"entries": list(single_cache.entries), "same_cache": model.pra_cache is single_cache}

## Factory Behavior

In [ ]:
resolver = create_resolver(resolver_config, documents, summaries)
prebuilt_cache = create_cache(cache_config)
cache = build_cache_from_metadata(model, dm.tokenizer, batch["metadata"], "cpu", resolver=resolver, cache=prebuilt_cache)

{
    "resolver_type": type(resolver).__name__,
    "cache_type": type(cache).__name__,
    "used_prebuilt_cache": cache is prebuilt_cache,
}

## Unsupported Services Fail Clearly

In [ ]:
errors = []
for factory, config in [
    (create_resolver, ResolverServiceConfig(type="unknown")),
    (create_cache, CacheServiceConfig(type="unknown")),
]:
    try:
        if factory is create_resolver:
            factory(config, {}, {})
        else:
            factory(config)
    except ValueError as exc:
        errors.append(str(exc))

errors